# Airbnb评论主题提炼 Agent

使用 LangGraph 构建 AI Agent，对 Airbnb 评论进行主题提炼，支持人机交互修改标签。

## 架构设计

### 图结构
```
review_loader → topic_analyzer ⇔ tools → result_formatter → END
```

### 节点说明
| 节点 | 功能 |
|------|------|
| review_loader | 加载评论，构建系统提示词 |
| topic_analyzer | LLM分析节点，提炼/修改标签 |
| tools | 工具执行器（update_tags / save_review）|
| result_formatter | 格式化输出结果 |

### 条件路由
- **should_use_tools**: LLM是否调用了工具 → tools 或 topic_analyzer
- **should_continue**: 工具类型 → update_tags→topic_analyzer（继续修改）| save_review→result_formatter（保存完成）

### Loop结构
- topic_analyzer → tools → topic_analyzer：人类可反复修改标签直到满意后保存

### 工具定义
| 工具 | 功能 |
|------|------|
| update_tags | 更新当前评论的主题标签 |
| save_review | 将评论和标签保存到txt文件 |

### MessageState类型
- SystemMessage: 系统提示词（角色定位+评论内容）
- HumanMessage: 用户输入（修改指令或确认）
- AIMessage: LLM响应和工具调用
- ToolMessage: 工具执行结果反馈

In [1]:
import csv
import json
import os
from typing import Annotated, Sequence, TypedDict
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode

# 环境配置
load_dotenv("env.env")
API_KEY = os.getenv("API_KEY")
API_BASE = os.getenv("API_BASE")
print(f"API配置: base_url={API_BASE}")
print(f"API_KEY: {API_KEY[:10]}...")

API配置: base_url=https://api.siliconflow.cn/v1
API_KEY: sk-avcysba...


In [2]:
# 加载评论数据
def load_reviews(filepath, num=20):
    reviews = []
    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            if i >= num:
                break
            comment = row.get("Comments", "").strip()
            if comment:
                reviews.append(comment)
    return reviews

reviews_data = load_reviews("reviews.csv", num=20)
print(f"加载了 {len(reviews_data)} 条评论")
for i, r in enumerate(reviews_data[:3]):
    print(f"\n评论 {i+1}: {r[:80]}...")

加载了 20 条评论

评论 1: The listing lived up to the name! we enjoyed the privacy of our own space, was c...

评论 2: My wife and I were able to relax, work, and enjoy Albany. The park across the st...

评论 3: The house was nice and clean. Parking is easy. The place is close to restaurants...


In [3]:
# 全局变量
current_review_global = ""
current_tags_global = []
all_results = []

# 状态定义
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

print("AgentState 定义完成")
print("字段: messages (Annotated[Sequence[BaseMessage], add_messages])")

AgentState 定义完成
字段: messages (Annotated[Sequence[BaseMessage], add_messages])


In [4]:
# 工具定义

@tool
def update_tags(tags: str) -> str:
    """更新当前评论的主题标签列表。"""
    global current_tags_global
    current_tags_global = [t.strip() for t in tags.split(",")]
    return f"标签已更新为: {current_tags_global}"

@tool
def save_review(filename: str) -> str:
    """将当前评论和标签保存到文本文件，并标记当前评论处理完成。"""
    global current_tags_global, current_review_global, all_results

    if not filename.endswith(".txt"):
        filename = f"{filename}.txt"

    result = {"review": current_review_global, "tags": list(current_tags_global)}
    all_results.append(result)

    with open(filename, "a", encoding="utf-8") as f:
        f.write(json.dumps(result, ensure_ascii=False) + "\n\n")

    return f"已保存到 {filename}！标签: {current_tags_global}"

tools_list = [update_tags, save_review]
print(f"工具定义完成: {[t.name for t in tools_list]}")

工具定义完成: ['update_tags', 'save_review']


In [5]:
# LLM 配置
model = ChatOpenAI(
    model="Qwen/Qwen2.5-72B-Instruct",
    api_key=API_KEY,
    base_url=API_BASE,
    temperature=0.3
).bind_tools(tools_list)

print("LLM 配置完成: Qwen/Qwen2.5-72B-Instruct")

LLM 配置完成: Qwen/Qwen2.5-72B-Instruct


In [8]:
# 节点函数
SYSTEM_PROMPT_TEMPLATE = """你是一个Airbnb评论主题提炼助手。你的任务是：
1. 阅读评论内容，提炼出2-5个主题标签
2. 立即使用 update_tags 工具设置标签
3. 展示当前标签并等待人类反馈
4. 如果人类要求修改标签，使用 update_tags 工具更新标签
5. 当人类输入“保存”、“确认”、“好的”、“ok”或类似确认词时，使用 save_review 工具保存结果，文件名固定为 review_results

标签要求：
- 用中文描述，简洁（2-6个字）
- 反映评论中提到的具体方面（如“干净舒适”、“设施齐全”、“位置便利”、“房东热情”等）
- 每次只调用一个工具

当前评论内容：{review}"""

def review_loader(state: AgentState) -> AgentState:
    """加载评论，构建系统提示词和初始人类消息"""
    global current_tags_global
    current_tags_global = []

    system_prompt = SystemMessage(content=SYSTEM_PROMPT_TEMPLATE.format(review=current_review_global))

    human_message = HumanMessage(
        content=f"请分析以下Airbnb评论，提炼出主题标签：\n\n{current_review_global}"
    )

    print(f"\n{'─'*60}")
    print(f"评论内容: {current_review_global[:100]}...")
    print(f"\n{'─'*60}")

    return {"messages": [system_prompt, human_message]}


def topic_analyzer(state: AgentState) -> AgentState:
    """LLM分析节点：分析评论提炼标签，或根据人类反馈修改标签"""
    messages = list(state["messages"])

    # 非首次调用时，询问人类反馈
    if len(messages) > 2:
        last_msg = messages[-1]
        if isinstance(last_msg, ToolMessage) and last_msg.name == "update_tags":
            print(f"\n  当前标签: {current_tags_global}")
            user_input = input(
                "\n  请输入指令（修改标签如'添加XX'、'删除XX'；直接回车保存）: "
            )
            if not user_input.strip():
                user_input = "保存"
            print(f"  用户: {user_input}")
            messages.append(HumanMessage(content=user_input))
        elif isinstance(last_msg, AIMessage):
            user_input = input(
                "\n  请输入指令（修改标签如'添加XX'、'删除XX'；直接回车保存）: "
            )
            if not user_input.strip():
                user_input = "保存"
            print(f"  用户: {user_input}")
            messages.append(HumanMessage(content=user_input))

    response = model.invoke(messages)

    print(f"\n  AI: {response.content}")
    if hasattr(response, "tool_calls") and response.tool_calls:
        print(f"调用工具: {[tc['name'] for tc in response.tool_calls]}")

    return {"messages": [response]}


def should_use_tools(state: AgentState) -> str:
    """条件路由：判断LLM是否调用了工具"""
    messages = state["messages"]
    if not messages:
        return "topic_analyzer"
    last_msg = messages[-1]
    if isinstance(last_msg, AIMessage) and hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        return "tools"
    return "topic_analyzer"


def should_continue(state: AgentState) -> str:
    """条件路由：继续修改标签还是保存并进入下一条评论"""
    messages = state["messages"]
    for msg in reversed(messages):
        if isinstance(msg, ToolMessage):
            if msg.name == "save_review":
                return "result_formatter"
            elif msg.name == "update_tags":
                return "topic_analyzer"
            break
    return "topic_analyzer"


def result_formatter(state: AgentState) -> AgentState:
    """格式化输出当前评论的处理结果"""
    print(f"\n  ✓ 本条评论处理完成！")
    print(f"    评论: {current_review_global[:60]}...")
    print(f"    标签: {current_tags_global}")
    return state


print("节点函数定义完成")

节点函数定义完成


In [9]:
# 图构建
graph = StateGraph(AgentState)

graph.add_node("review_loader", review_loader)
graph.add_node("topic_analyzer", topic_analyzer)
graph.add_node("tools", ToolNode(tools_list))
graph.add_node("result_formatter", result_formatter)

graph.set_entry_point("review_loader")

graph.add_edge("review_loader", "topic_analyzer")

graph.add_conditional_edges(
    "topic_analyzer",
    should_use_tools,
    {"tools": "tools", "topic_analyzer": "topic_analyzer"},
)

graph.add_conditional_edges(
    "tools",
    should_continue,
    {"topic_analyzer": "topic_analyzer", "result_formatter": "result_formatter"},
)

graph.add_edge("result_formatter", END)

app = graph.compile()

print("图构建完成！")
print(f"节点: {list(graph.nodes.keys())}")
print(f"入口: review_loader")
print(f"\n图结构:")
print("  review_loader → topic_analyzer ⇔ tools → result_formatter → END")

图构建完成！
节点: ['review_loader', 'topic_analyzer', 'tools', 'result_formatter']
入口: review_loader

图结构:
  review_loader → topic_analyzer ⇔ tools → result_formatter → END


In [10]:
# 处理单条评论
def process_single_review(review_text):
    """处理单条评论"""
    global current_review_global, current_tags_global
    current_review_global = review_text
    current_tags_global = []
    
    state = {"messages": []}
    for step in app.stream(state, stream_mode="values"):
        pass
    
    return {"review": current_review_global, "tags": list(current_tags_global)}

print("处理函数定义完成")

处理函数定义完成


In [ ]:
# 运行Agent处理所有评论
def run_review_agent(num_reviews=20):
    """运行Agent处理指定数量的评论"""
    global all_results
    
    reviews = load_reviews("reviews.csv", num=num_reviews)
    all_results = []
    
    # 清空已有的结果文件
    if os.path.exists("review_results.txt"):
        os.remove("review_results.txt")
    
    print(f"\n{'='*60}")
    print("Airbnb 评论主题提炼 Agent")
    print(f" 将处理 {len(reviews)} 条评论")
    print(f"{'='*60}")
    
    for i, review in enumerate(reviews):
        print(f"\n{'#'*60}")
        print(f"  正在处理第 {i+1}/{len(reviews)} 条评论")
        print(f"\n{'#'*60}")
        process_single_review(review)
    
    # 打印最终结果
    print(f"\n\n{'='*60}")
    print("所有评论处理完成！最终结果：")
    print(f"\n{'='*60}")
    for r in all_results:
        review_preview = r["review"][:80]
        print(f'\n  评论: {review_preview}...')
        print(f'  标签: {r["tags"]}')
    
    return all_results

print("运行函数定义完成")

运行函数定义完成


In [13]:
# 执行！处理20条评论（每条评论会要求人工确认，直接回车即可保存）
# 如需处理较少评论用于测试，可修改 num_reviews 参数
results = run_review_agent(num_reviews=20)


  Airbnb评论主题提炼 Agent
  将处理 20 条评论

############################################################
  正在处理第 1/20 条评论

############################################################

────────────────────────────────────────────────────────────
评论内容: The listing lived up to the name! we enjoyed the privacy of our own space, was clean, inviting, and ...

────────────────────────────────────────────────────────────

  AI: 
调用工具: ['update_tags']

  当前标签: ['隐私空间', '干净舒适', '设施齐全', '房东热情', '位置安静']
  用户: 删除标签隐私空间

  AI: 
调用工具: ['update_tags']

  当前标签: ['干净舒适', '设施齐全', '房东热情', '位置安静']
  用户: 保存

  AI: 
调用工具: ['save_review']

  ✓ 本条评论处理完成！
    评论: The listing lived up to the name! we enjoyed the privacy of ...
    标签: ['干净舒适', '设施齐全', '房东热情', '位置安静']

############################################################
  正在处理第 2/20 条评论

############################################################

────────────────────────────────────────────────────────────
评论内容: My wife and I were able to relax, work, and enj

## 最终结果

打印 review_results.txt 文件内容：

In [14]:
# 打印最终形成的txt文档内容
try:
    with open("review_results.txt", "r", encoding="utf-8") as f:
        content = f.read()
    print(content)
except FileNotFoundError:
    print("文件未找到，请先运行上面的Agent")

{"review": "The listing lived up to the name! we enjoyed the privacy of our own space, was clean, inviting, and Elsa provided many amenities for our stay. she responded quickly to any questions, and location was in a quiet residential neighborhood. would highly recommend!", "tags": ["干净舒适", "设施齐全", "房东热情", "位置安静"]}

{"review": "My wife and I were able to relax, work, and enjoy Albany. The park across the street was a great treat for family outings with our Dog! The host's were extremely flexible when we needed to adjust out reservation. Location was perfect!!", "tags": ["放松舒适", "位置便利", "房东灵活", "宠物友好"]}

{"review": "The house was nice and clean. Parking is easy. The place is close to restaurants. Kitchen had everything You need to cook. I wish there was a regular coffee machine instead of Chemex pour-over. Will stay again!", "tags": ["干净舒适", "停车方便", "位置便利", "厨房齐全", "咖啡机需求"]}

{"review": "Great place and location!  Nice and cool in the heat as well", "tags": ["位置便利", "环境舒适", "温度适宜"]}

{"